# Subject-Independent Random Forest — Mean + Variance

Evaluation protocol reproduced from `GNN2.ipynb` (DGCNN) so that RF and DGCNN
results are directly comparable on the **same held-out test subjects**.

**Reference sources**
- Subject split, label binarisation, band definitions, PSD method, channel-ranking
  procedure: taken from the DGCNN notebook.
- Feature extraction (Mean + Variance): taken from the original RF notebooks.

**Documented deviations from the written spec** (see notebook end for rationale):
1. Channel ranking uses **band-power** importance in *both* notebooks, matching the
   DGCNN, so all three models are evaluated on identical channel subsets.
   Set `RANK_ON_OWN_FEATURES = True` in the config cell to instead rank on this
   notebook's own feature representation (spec Part 6 as literally written).
2. The channel-ranking RF uses `n_estimators=200` (DGCNN value); the classifier
   RF uses `n_estimators=100` (spec Part 8). They are separate models.
3. Best-configuration selection uses Balanced Accuracy -> F1 -> Accuracy (spec
   Part 14). The DGCNN notebook selected on Accuracy alone; re-derive the DGCNN
   best-config table with this same priority before tabulating them side by side.


In [1]:
# ============================================================
# SECTION 1 — IMPORTS
# ============================================================
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import welch

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except ImportError:
    HAS_SGKF = False

warnings.filterwarnings("ignore", category=UserWarning)
print("StratifiedGroupKFold available:", HAS_SGKF)


StratifiedGroupKFold available: True


In [2]:
# ============================================================
# SECTION 2 — CONFIGURATION
# All values below are copied from the DGCNN notebook (GNN2.ipynb)
# unless explicitly noted.
# ============================================================

EEG_PATH    = "eeg_ml.npy"
LABELS_PATH = "labels_ml.npy"

OUTPUT_DIR = "rf_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FEATURE_NAME = "Mean + Variance"
FEATURE_TAG  = "meanvar"          # used in output filenames

FS               = 128                  # DEAP sampling rate (Hz)  [DGCNN]
N_SUBJECTS       = 32
N_TRIALS_PER_SUB = 40
N_CHANNELS       = 32
N_TIMEPOINTS     = 8064

RANDOM_STATE = 42                       # [DGCNN]

EMOTIONS       = {"Valence": 0, "Arousal": 1, "Dominance": 2, "Liking": 3}
CHANNEL_COUNTS = [32, 16, 8, 4]

# Frequency bands — identical to the DGCNN notebook.
BANDS = {
    "Delta": (0.5, 4), "Theta": (4, 8), "Alpha": (8, 13),
    "Beta": (13, 30), "Gamma": (30, 45),
}

# --- Random Forest settings -------------------------------------------------
# Classifier RF: spec Part 8.
RF_PARAMS = dict(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# Channel-ranking RF: DGCNN used n_estimators=200. Kept at 200 so the ranked
# channel subsets reproduce the DGCNN's exactly. This is a *different model*
# from the classifier above and does not contradict spec Part 8.
RANKER_PARAMS = dict(
    n_estimators=200,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# --- Channel-ranking feature basis ------------------------------------------
# False (default) -> rank channels on BAND-POWER importance, matching the DGCNN,
#                    so RF and DGCNN are compared on identical channel subsets.
# True            -> rank channels on this notebook's own features (spec Part 6
#                    as literally written). Channel subsets will then differ
#                    between the two RF notebooks and from the DGCNN.
RANK_ON_OWN_FEATURES = False

# Importance aggregation across a channel's features: SUM.
# (Matches the DGCNN's rank_channels_by_rf_importance. Note that sum and mean
# produce an identical ranking here because every channel contributes the same
# number of features.)
IMPORTANCE_AGGREGATION = "sum"

# --- Cross-validation -------------------------------------------------------
CV_FOLDS = 5    # run strictly within the 20 training subjects

print(f"Feature representation : {FEATURE_NAME}")
print(f"Importance aggregation : {IMPORTANCE_AGGREGATION}")
print(f"Rank on own features   : {RANK_ON_OWN_FEATURES}")
print(f"Output directory       : {OUTPUT_DIR}/")


Feature representation : Mean + Variance
Importance aggregation : sum
Rank on own features   : False
Output directory       : rf_outputs/


In [3]:
# ============================================================
# SECTION 3 — LOAD EEG AND LABELS
# The already-preprocessed arrays are used as-is. No re-preprocessing.
# ============================================================

def load_data(eeg_path=EEG_PATH, labels_path=LABELS_PATH):
    eeg = np.load(eeg_path)
    labels = np.load(labels_path)

    # Accept either the flat (1280, 32, 8064) layout or the DGCNN's
    # (32, 40, 32, 8064) layout, reshaping the latter exactly as the DGCNN does.
    if eeg.ndim == 4:
        print(f"EEG loaded as {eeg.shape}; reshaping to flat trial layout.")
        eeg = eeg.reshape(-1, eeg.shape[2], eeg.shape[3])
    if labels.ndim == 3:
        labels = labels.reshape(-1, labels.shape[2])

    eeg = np.ascontiguousarray(eeg, dtype=np.float32)

    print("EEG shape    :", eeg.shape)
    print("Labels shape :", labels.shape)
    return eeg, labels


eeg, labels = load_data()

assert eeg.shape == (N_SUBJECTS * N_TRIALS_PER_SUB, N_CHANNELS, N_TIMEPOINTS), (
    f"Unexpected EEG shape {eeg.shape}; expected "
    f"{(N_SUBJECTS * N_TRIALS_PER_SUB, N_CHANNELS, N_TIMEPOINTS)}"
)
assert labels.shape == (N_SUBJECTS * N_TRIALS_PER_SUB, 4), (
    f"Unexpected labels shape {labels.shape}"
)
print("\nShape assertions passed.")


EEG shape    : (1280, 32, 8064)
Labels shape : (1280, 4)

Shape assertions passed.


In [4]:
# ============================================================
# SECTION 4 — SUBJECT IDS
# Trials 0-39 -> subject 0, 40-79 -> subject 1, ... (DGCNN ordering)
# ============================================================

subject_ids = np.repeat(np.arange(N_SUBJECTS), N_TRIALS_PER_SUB)

assert subject_ids.shape == (1280,), subject_ids.shape
assert len(subject_ids) == len(eeg)

counts = np.bincount(subject_ids, minlength=N_SUBJECTS)
print("Subject IDs shape:", subject_ids.shape)
print("Trials per subject (all should be 40):")
for s in range(N_SUBJECTS):
    print(f"  Subject {s:2d}: {counts[s]} trials")
assert np.all(counts == N_TRIALS_PER_SUB), "Uneven trials per subject"
print("\nAll 32 subjects have exactly 40 trials.")


Subject IDs shape: (1280,)
Trials per subject (all should be 40):
  Subject  0: 40 trials
  Subject  1: 40 trials
  Subject  2: 40 trials
  Subject  3: 40 trials
  Subject  4: 40 trials
  Subject  5: 40 trials
  Subject  6: 40 trials
  Subject  7: 40 trials
  Subject  8: 40 trials
  Subject  9: 40 trials
  Subject 10: 40 trials
  Subject 11: 40 trials
  Subject 12: 40 trials
  Subject 13: 40 trials
  Subject 14: 40 trials
  Subject 15: 40 trials
  Subject 16: 40 trials
  Subject 17: 40 trials
  Subject 18: 40 trials
  Subject 19: 40 trials
  Subject 20: 40 trials
  Subject 21: 40 trials
  Subject 22: 40 trials
  Subject 23: 40 trials
  Subject 24: 40 trials
  Subject 25: 40 trials
  Subject 26: 40 trials
  Subject 27: 40 trials
  Subject 28: 40 trials
  Subject 29: 40 trials
  Subject 30: 40 trials
  Subject 31: 40 trials

All 32 subjects have exactly 40 trials.


In [5]:
# ============================================================
# SECTION 5 — EXACT 20 / 5 / 7 SUBJECT SPLIT
#
# These IDs are hard-coded rather than regenerated, but they are exactly what
# the DGCNN's subject_wise_split() produces:
#     train_val, test = train_test_split(arange(32), test_size=0.2, random_state=42)
#     train, val      = train_test_split(train_val, test_size=0.2, random_state=42)
# The assertion below re-derives them with sklearn and confirms the match, so a
# different sklearn version cannot silently desynchronise RF from DGCNN.
# ============================================================

TRAIN_SUBJECTS = np.array([0, 1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 13,
                           14, 16, 20, 21, 22, 27, 28, 31])
VAL_SUBJECTS   = np.array([18, 19, 23, 25, 26])
TEST_SUBJECTS  = np.array([8, 9, 15, 17, 24, 29, 30])

# --- Verify these reproduce the DGCNN's generated split ---------------------
from sklearn.model_selection import train_test_split as _tts

_tv, _te = _tts(np.arange(N_SUBJECTS), test_size=0.2, random_state=RANDOM_STATE)
_tr, _va = _tts(_tv, test_size=0.2, random_state=RANDOM_STATE)

if not (sorted(_tr) == sorted(TRAIN_SUBJECTS)
        and sorted(_va) == sorted(VAL_SUBJECTS)
        and sorted(_te) == sorted(TEST_SUBJECTS)):
    raise RuntimeError(
        "Hard-coded split does NOT match the DGCNN's generated split on this "
        "sklearn version. RF and DGCNN would no longer be comparable.\n"
        f"  Regenerated train: {sorted(_tr)}\n"
        f"  Regenerated val  : {sorted(_va)}\n"
        f"  Regenerated test : {sorted(_te)}"
    )
print("Hard-coded split matches the DGCNN's generated split exactly.\n")

train_mask = np.isin(subject_ids, TRAIN_SUBJECTS)
val_mask   = np.isin(subject_ids, VAL_SUBJECTS)
test_mask  = np.isin(subject_ids, TEST_SUBJECTS)

train_idx = np.where(train_mask)[0]
val_idx   = np.where(val_mask)[0]
test_idx  = np.where(test_mask)[0]

print(f"Train subjects ({len(TRAIN_SUBJECTS)}):", TRAIN_SUBJECTS.tolist())
print(f"Val subjects   ({len(VAL_SUBJECTS)}):", VAL_SUBJECTS.tolist())
print(f"Test subjects  ({len(TEST_SUBJECTS)}):", TEST_SUBJECTS.tolist())
print()
print(f"Train trials : {train_mask.sum()}")
print(f"Val trials   : {val_mask.sum()}")
print(f"Test trials  : {test_mask.sum()}")


Hard-coded split matches the DGCNN's generated split exactly.

Train subjects (20): [0, 1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 14, 16, 20, 21, 22, 27, 28, 31]
Val subjects   (5): [18, 19, 23, 25, 26]
Test subjects  (7): [8, 9, 15, 17, 24, 29, 30]

Train trials : 800
Val trials   : 200
Test trials  : 280


In [6]:
# ============================================================
# SECTION 6 — VERIFY SPLIT (spec Part 2)
# ============================================================

assert len(TRAIN_SUBJECTS) == 20, "Train must have 20 subjects"
assert len(VAL_SUBJECTS)   == 5,  "Validation must have 5 subjects"
assert len(TEST_SUBJECTS)  == 7,  "Test must have 7 subjects"

assert train_mask.sum() == 800, f"Train trials = {train_mask.sum()}, expected 800"
assert val_mask.sum()   == 200, f"Val trials = {val_mask.sum()}, expected 200"
assert test_mask.sum()  == 280, f"Test trials = {test_mask.sum()}, expected 280"

# No subject appears in more than one split
assert not set(TRAIN_SUBJECTS) & set(VAL_SUBJECTS),  "Train/Val subject overlap"
assert not set(TRAIN_SUBJECTS) & set(TEST_SUBJECTS), "Train/Test subject overlap"
assert not set(VAL_SUBJECTS)   & set(TEST_SUBJECTS), "Val/Test subject overlap"

# No trial index appears in more than one split
assert not set(train_idx) & set(val_idx),  "Train/Val trial overlap"
assert not set(train_idx) & set(test_idx), "Train/Test trial overlap"
assert not set(val_idx)   & set(test_idx), "Val/Test trial overlap"

# Splits are exhaustive and non-overlapping
assert len(train_idx) + len(val_idx) + len(test_idx) == 1280

print("All split assertions passed.")
print("  20 train / 5 val / 7 test subjects")
print("  800 train / 200 val / 280 test trials")
print("  No subject overlap, no trial overlap.")

# Fingerprint of the test trials — compare this against the DGCNN run to prove
# both models were scored on the same 280 trials (spec Part 18, checks 11-12).
TEST_TRIAL_FINGERPRINT = json.dumps(sorted(test_idx.tolist()))
print("\nTest-trial fingerprint (first 10):", sorted(test_idx.tolist())[:10], "...")


All split assertions passed.
  20 train / 5 val / 7 test subjects
  800 train / 200 val / 280 test trials
  No subject overlap, no trial overlap.

Test-trial fingerprint (first 10): [320, 321, 322, 323, 324, 325, 326, 327, 328, 329] ...


In [7]:
# ============================================================
# SECTION 7 — LABEL PROCESSING
# Binarisation copied verbatim from the DGCNN:
#     y = (labels[:, col_idx] > 5).astype(int)
# Low = score <= 5, High = score > 5. Threshold NOT changed.
# ============================================================

def binarize(labels_array, col_idx):
    return (labels_array[:, col_idx] > 5).astype(int)


binary_labels = {}

for emotion_name, col_idx in EMOTIONS.items():
    raw = labels[:, col_idx]
    y = binarize(labels, col_idx)
    binary_labels[emotion_name] = y

    print("=" * 62)
    print(f"{emotion_name.upper()}  (label column {col_idx})")
    print("=" * 62)
    print(f"  Raw  -> min {raw.min():.2f}  max {raw.max():.2f}  "
          f"mean {raw.mean():.3f}  std {raw.std():.3f}  median {np.median(raw):.2f}")

    for split_name, m in (("Overall", np.ones(1280, bool)),
                          ("Train  ", train_mask),
                          ("Val    ", val_mask),
                          ("Test   ", test_mask)):
        c = np.bincount(y[m], minlength=2)
        n = int(m.sum())
        print(f"  {split_name} (n={n:4d}) -> Low {c[0]:4d} ({c[0]/n:.1%})  "
              f"High {c[1]:4d} ({c[1]/n:.1%})")
    print()


VALENCE  (label column 0)
  Raw  -> min 1.00  max 9.00  mean 5.254  std 2.130  median 5.04
  Overall (n=1280) -> Low  572 (44.7%)  High  708 (55.3%)
  Train   (n= 800) -> Low  360 (45.0%)  High  440 (55.0%)
  Val     (n= 200) -> Low   80 (40.0%)  High  120 (60.0%)
  Test    (n= 280) -> Low  132 (47.1%)  High  148 (52.9%)

AROUSAL  (label column 1)
  Raw  -> min 1.00  max 9.00  mean 5.157  std 2.020  median 5.23
  Overall (n=1280) -> Low  543 (42.4%)  High  737 (57.6%)
  Train   (n= 800) -> Low  356 (44.5%)  High  444 (55.5%)
  Val     (n= 200) -> Low   65 (32.5%)  High  135 (67.5%)
  Test    (n= 280) -> Low  122 (43.6%)  High  158 (56.4%)

DOMINANCE  (label column 2)
  Raw  -> min 1.00  max 9.00  mean 5.383  std 2.096  median 5.24
  Overall (n=1280) -> Low  500 (39.1%)  High  780 (60.9%)
  Train   (n= 800) -> Low  323 (40.4%)  High  477 (59.6%)
  Val     (n= 200) -> Low   71 (35.5%)  High  129 (64.5%)
  Test    (n= 280) -> Low  106 (37.9%)  High  174 (62.1%)

LIKING  (label column 3)
 

In [8]:
# ============================================================
# SECTION 8 — FEATURE EXTRACTION: MEAN + VARIANCE
#
# Per selected channel: mean and variance over the full 8064-sample trial.
# The 3 s baseline is NOT removed, matching the DGCNN, which windows all
# 8064 samples. Feature ordering is [ch0_mean, ch0_var, ch1_mean, ch1_var, ...].
#
# Features = n_channels x 2:
#   32 ch -> 64 | 16 ch -> 32 | 8 ch -> 16 | 4 ch -> 8
# ============================================================

N_FEATURES_PER_CHANNEL = 2
FEATURE_NAMES_PER_CHANNEL = ["Mean", "Variance"]


def extract_features(eeg_data):
    """(n_trials, n_channels, n_time) -> (n_trials, n_channels * 2)."""
    mean = eeg_data.mean(axis=2)
    var = eeg_data.var(axis=2)
    # Interleave so each channel's features are contiguous.
    feats = np.empty((eeg_data.shape[0], eeg_data.shape[1] * 2), dtype=np.float64)
    feats[:, 0::2] = mean
    feats[:, 1::2] = var
    return feats


# Sanity check on a small slice
_probe = extract_features(eeg[:4])
assert _probe.shape == (4, N_CHANNELS * N_FEATURES_PER_CHANNEL), _probe.shape
assert np.isclose(_probe[0, 0], eeg[0, 0].mean())
assert np.isclose(_probe[0, 1], eeg[0, 0].var())
print(f"Mean+Variance extractor OK -> {_probe.shape[1]} features for 32 channels")


Mean+Variance extractor OK -> 64 features for 32 channels


In [9]:
# ============================================================
# SECTION 8b — PRECOMPUTE FULL 32-CHANNEL FEATURES
#
# Features are computed per-trial and independently per-channel, so a channel
# subset is just a column subset of the full matrix. Computing once avoids
# recomputing for every emotion x channel-count combination.
#
# IMPORTANT: computing features for all 1280 trials here is NOT leakage. No
# model is fitted at this point, and every feature is a function of a single
# trial only — no cross-trial statistic (no shared scaler, no shared PSD
# normalisation) is involved. Fitting is confined to training rows throughout.
# ============================================================

print(f"Extracting Mean + Variance features for all 1280 trials...")
X_full = extract_features(eeg)
print("Full feature matrix:", X_full.shape)

assert X_full.shape == (1280, N_CHANNELS * N_FEATURES_PER_CHANNEL)
assert np.isfinite(X_full).all(), "Non-finite values in feature matrix"


def channel_columns(channel_idx):
    """Column indices for the given channels, preserving ranking order."""
    return np.concatenate([
        np.arange(ch * N_FEATURES_PER_CHANNEL, (ch + 1) * N_FEATURES_PER_CHANNEL)
        for ch in channel_idx
    ])


# Band-power matrix used for channel ranking when RANK_ON_OWN_FEATURES is False.
if RANK_ON_OWN_FEATURES:
    X_rank_basis = X_full
    RANK_FEATURES_PER_CHANNEL = N_FEATURES_PER_CHANNEL
    RANK_BASIS_NAME = FEATURE_NAME
else:
    if "meanvar" == "bandpower":
        X_rank_basis = X_full
    else:
        print("Extracting band-power features for DGCNN-matched channel ranking...")

        def _bp(psd, freqs, fmin, fmax):
            idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
            return np.mean(psd[idx]) if idx.size > 0 else 0.0

        _n = eeg.shape[0]
        X_rank_basis = np.zeros((_n, N_CHANNELS * len(BANDS)))
        for i, sample in enumerate(eeg):
            vec = []
            for ch in range(N_CHANNELS):
                f_, p_ = welch(sample[ch], fs=FS, nperseg=FS * 2)
                for fmin, fmax in BANDS.values():
                    vec.append(_bp(p_, f_, fmin, fmax))
            X_rank_basis[i] = vec
    RANK_FEATURES_PER_CHANNEL = len(BANDS)
    RANK_BASIS_NAME = "Band-Power (DGCNN-matched)"

print(f"Channel-ranking basis: {RANK_BASIS_NAME} -> {X_rank_basis.shape}")


Extracting Mean + Variance features for all 1280 trials...
Full feature matrix: (1280, 64)
Extracting band-power features for DGCNN-matched channel ranking...
Channel-ranking basis: Band-Power (DGCNN-matched) -> (1280, 160)


In [10]:
# ============================================================
# SECTION 9 — CHANNEL RANKING (TRAINING SUBJECTS ONLY)
#
# Reproduces the DGCNN's rank_channels_by_rf_importance:
#   RF(n_estimators=200, class_weight='balanced', random_state=42) fitted on
#   TRAINING-SUBJECT rows only; per-channel importance = SUM over that
#   channel's features; channels sorted best-first.
#
# Validation and test subjects are never seen here.
# ============================================================

def rank_channels(X_basis, y, train_row_mask, features_per_channel):
    X_tr = X_basis[train_row_mask]
    y_tr = y[train_row_mask]

    # Hard guard: ranking must only ever see 800 training rows.
    assert X_tr.shape[0] == 800, f"Ranking saw {X_tr.shape[0]} rows, expected 800"

    rf = RandomForestClassifier(**RANKER_PARAMS)
    rf.fit(X_tr, y_tr)

    imp = rf.feature_importances_
    scores = {}
    for ch in range(N_CHANNELS):
        start = ch * features_per_channel
        block = imp[start:start + features_per_channel]
        scores[ch] = block.sum() if IMPORTANCE_AGGREGATION == "sum" else block.mean()

    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked, scores


channel_rankings = {}
channel_subsets = {}
ranking_records = []

for emotion_name in EMOTIONS:
    y = binary_labels[emotion_name]
    ranked, scores = rank_channels(
        X_rank_basis, y, train_mask, RANK_FEATURES_PER_CHANNEL
    )
    channel_rankings[emotion_name] = ranked
    channel_subsets[emotion_name] = {n: ranked[:n] for n in CHANNEL_COUNTS}

    print("=" * 62)
    print(f"CHANNEL RANKING — {emotion_name.upper()}"
          f"  (basis: {RANK_BASIS_NAME}, aggregation: {IMPORTANCE_AGGREGATION})")
    print("=" * 62)
    print(f"{'Rank':>4}  {'Channel':>7}  {'Importance':>11}")
    for rank, ch in enumerate(ranked, start=1):
        print(f"{rank:>4}  {ch:>7}  {scores[ch]:>11.6f}")
        ranking_records.append({
            "Emotion": emotion_name,
            "Rank": rank,
            "Channel": ch,
            "Importance": round(float(scores[ch]), 6),
        })
    print()
    for n in CHANNEL_COUNTS:
        print(f"  Top-{n:2d}: {ranked[:n]}")
    print()

ranking_df = pd.DataFrame(ranking_records)
ranking_path = f"{OUTPUT_DIR}/subject_independent_{FEATURE_TAG}_channel_ranking.csv"
ranking_df.to_csv(ranking_path, index=False)
print(f"Saved channel rankings -> {ranking_path}")


CHANNEL RANKING — VALENCE  (basis: Band-Power (DGCNN-matched), aggregation: sum)
Rank  Channel   Importance
   1        5     0.037336
   2       15     0.034895
   3       13     0.034412
   4       23     0.034329
   5       29     0.033995
   6        3     0.033891
   7       19     0.033835
   8       28     0.033298
   9        0     0.033050
  10       18     0.032940
  11       31     0.032578
  12       26     0.032543
  13       11     0.031616
  14       25     0.031469
  15       10     0.030701
  16       24     0.030677
  17       21     0.030592
  18       17     0.030251
  19       12     0.030033
  20        7     0.030031
  21       30     0.029879
  22       14     0.029556
  23        1     0.029503
  24        9     0.029365
  25       22     0.029293
  26       16     0.029191
  27        8     0.029028
  28       20     0.028888
  29        6     0.028864
  30        2     0.028598
  31       27     0.027736
  32        4     0.027626

  Top-32: [5, 15, 13, 23, 2

In [11]:
# ============================================================
# SECTION 10 — CHANNEL SUBSET VERIFICATION (spec Part 18, checks 13-15)
#
# Note: these are nested by construction (all four subsets are prefixes of one
# ranked list), so passing them confirms the code does what it says but does
# NOT independently evidence absence of leakage. The leakage guarantee comes
# from the 800-row assertion inside rank_channels().
# ============================================================

for emotion_name, subsets in channel_subsets.items():
    assert set(subsets[4]).issubset(subsets[8]),   f"{emotion_name}: top-4 not in top-8"
    assert set(subsets[8]).issubset(subsets[16]),  f"{emotion_name}: top-8 not in top-16"
    assert set(subsets[16]).issubset(subsets[32]), f"{emotion_name}: top-16 not in top-32"
    assert sorted(subsets[32]) == list(range(N_CHANNELS)), \
        f"{emotion_name}: top-32 is not all 32 channels"
    for n in CHANNEL_COUNTS:
        assert len(set(subsets[n])) == n, f"{emotion_name}: duplicate channels in top-{n}"

print("Channel subset nesting verified for all four emotions.")


Channel subset nesting verified for all four emotions.


In [12]:
# ============================================================
# SECTION 11 — SUBJECT-AWARE CROSS-VALIDATION
#
# CV runs strictly inside the 20 training subjects. StratifiedGroupKFold keeps
# all of a subject's trials in one fold while balancing classes; GroupKFold is
# the fallback on older scikit-learn. Ordinary StratifiedKFold is NOT used —
# it would split a subject across folds.
#
# CV numbers are reported in their own columns and are NEVER used as test
# metrics.
# ============================================================

def subject_aware_cv(X, y, groups, n_splits=CV_FOLDS):
    """Mean/std CV accuracy over training subjects only."""
    # Hard guard: CV must never see a validation or test subject.
    assert not set(np.unique(groups)) & set(VAL_SUBJECTS.tolist()), \
        "CV groups contain validation subjects"
    assert not set(np.unique(groups)) & set(TEST_SUBJECTS.tolist()), \
        "CV groups contain test subjects"

    if HAS_SGKF:
        splitter = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE
        )
        split_iter = splitter.split(X, y, groups)
    else:
        splitter = GroupKFold(n_splits=n_splits)
        split_iter = splitter.split(X, y, groups)

    scores = []
    for tr, te in split_iter:
        # Fold must contain both classes to be meaningful.
        if len(np.unique(y[tr])) < 2:
            continue
        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X[tr], y[tr])
        scores.append(accuracy_score(y[te], model.predict(X[te])))

    if not scores:
        return float("nan"), float("nan")
    return float(np.mean(scores)), float(np.std(scores))


print("CV method:", "StratifiedGroupKFold" if HAS_SGKF else "GroupKFold",
      f"({CV_FOLDS} folds, training subjects only)")


CV method: StratifiedGroupKFold (5 folds, training subjects only)


In [13]:
# ============================================================
# SECTION 12-15 — TRAIN / VALIDATE / TEST + METRICS
#
# For each (emotion, channel count):
#   1. Subject-aware CV within the 20 training subjects.
#   2. Fit RF on the 800 training trials only.
#   3. Evaluate on the 200 validation trials (reporting / comparison only).
#   4. Evaluate on the 280 held-out test trials — a single, final fit-free pass.
#
# The model is never refitted on validation or test data.
# ============================================================

def compute_metrics(y_true, y_pred, y_prob):
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": auc,
    }


def run_experiment(emotion_name, channel_count, channel_idx, y):
    cols = channel_columns(channel_idx)

    X_tr, y_tr = X_full[train_mask][:, cols], y[train_mask]
    X_va, y_va = X_full[val_mask][:, cols],   y[val_mask]
    X_te, y_te = X_full[test_mask][:, cols],  y[test_mask]

    assert X_tr.shape[0] == 800 and X_va.shape[0] == 200 and X_te.shape[0] == 280
    assert X_tr.shape[1] == channel_count * N_FEATURES_PER_CHANNEL

    # --- 1. Subject-aware CV, training subjects only ---
    cv_mean, cv_std = subject_aware_cv(X_tr, y_tr, subject_ids[train_mask])

    # --- 2. Final fit on training trials only ---
    model = RandomForestClassifier(**RF_PARAMS)
    model.fit(X_tr, y_tr)

    # --- 3. Validation (model selection / reporting only) ---
    val_metrics = compute_metrics(
        y_va, model.predict(X_va), model.predict_proba(X_va)[:, 1]
    )

    # --- 4. Final held-out test evaluation ---
    test_pred = model.predict(X_te)
    test_prob = model.predict_proba(X_te)[:, 1]
    test_metrics = compute_metrics(y_te, test_pred, test_prob)
    cm = confusion_matrix(y_te, test_pred)

    print(f"  {emotion_name:>10} | {channel_count:2d} ch | "
          f"{channel_count * N_FEATURES_PER_CHANNEL:3d} feat | "
          f"CV {cv_mean:.4f}+/-{cv_std:.4f} | "
          f"Val Acc {val_metrics['Accuracy']:.4f} | "
          f"TEST Acc {test_metrics['Accuracy']:.4f} "
          f"BalAcc {test_metrics['Balanced Accuracy']:.4f} "
          f"F1 {test_metrics['F1-Score']:.4f}")

    row = {
        "Emotion": emotion_name,
        "Configuration": f"Top-{channel_count}",
        "Channels": channel_count,
        "Num Features": channel_count * N_FEATURES_PER_CHANNEL,
        "Channel Indices": json.dumps(list(channel_idx)),
    }
    # Final test metrics occupy the primary columns (spec Part 13).
    row.update({k: round(v, 4) for k, v in test_metrics.items()})
    row.update({f"Val {k}": round(v, 4) for k, v in val_metrics.items()})
    row["CV Mean"] = round(cv_mean, 4)
    row["CV Std"] = round(cv_std, 4)

    return row, cm, (y_te, test_pred)


results = []
confusion_matrices = {}
test_predictions = {}

for emotion_name in EMOTIONS:
    print("=" * 100)
    print(f"EMOTION: {emotion_name.upper()}")
    print("=" * 100)
    y = binary_labels[emotion_name]
    for ch_count in CHANNEL_COUNTS:
        ch_idx = channel_subsets[emotion_name][ch_count]
        row, cm, preds = run_experiment(emotion_name, ch_count, ch_idx, y)
        results.append(row)
        confusion_matrices[(emotion_name, ch_count)] = cm
        test_predictions[(emotion_name, ch_count)] = preds
    print()

results_df = pd.DataFrame(results)
print(f"Completed {len(results_df)} experiments "
      f"({len(EMOTIONS)} emotions x {len(CHANNEL_COUNTS)} configurations).")
assert len(results_df) == 16, f"Expected 16 rows, got {len(results_df)}"


EMOTION: VALENCE
     Valence | 32 ch |  64 feat | CV 0.5262+/-0.0329 | Val Acc 0.5450 | TEST Acc 0.5679 BalAcc 0.5499 F1 0.6790
     Valence | 16 ch |  32 feat | CV 0.5262+/-0.0443 | Val Acc 0.5500 | TEST Acc 0.5679 BalAcc 0.5462 F1 0.6937
     Valence |  8 ch |  16 feat | CV 0.4963+/-0.0255 | Val Acc 0.5150 | TEST Acc 0.5143 BalAcc 0.4922 F1 0.6566
     Valence |  4 ch |   8 feat | CV 0.5038+/-0.0406 | Val Acc 0.4750 | TEST Acc 0.4786 BalAcc 0.4601 F1 0.6138

EMOTION: AROUSAL
     Arousal | 32 ch |  64 feat | CV 0.5137+/-0.0801 | Val Acc 0.5650 | TEST Acc 0.5000 BalAcc 0.4813 F1 0.5858
     Arousal | 16 ch |  32 feat | CV 0.5337+/-0.0795 | Val Acc 0.6600 | TEST Acc 0.5143 BalAcc 0.5015 F1 0.5828
     Arousal |  8 ch |  16 feat | CV 0.5912+/-0.0653 | Val Acc 0.6100 | TEST Acc 0.4964 BalAcc 0.4875 F1 0.5552
     Arousal |  4 ch |   8 feat | CV 0.5737+/-0.0437 | Val Acc 0.6050 | TEST Acc 0.4714 BalAcc 0.4588 F1 0.5432

EMOTION: DOMINANCE
   Dominance | 32 ch |  64 feat | CV 0.5725+/-0.0

In [14]:
# ============================================================
# SECTION 18-19 — RESULTS TABLE AND CSV OUTPUT
#
# Column meaning (spec Part 13):
#   Accuracy / Balanced Accuracy / Precision / Recall / F1-Score / ROC-AUC
#       -> FINAL TEST metrics, 280 held-out trials.
#   Val *  -> validation metrics, 200 trials. Reporting only.
#   CV Mean / CV Std -> subject-aware CV inside the training subjects.
# CV values are never placed in the Accuracy column.
# ============================================================

TEST_COLS = ["Accuracy", "Balanced Accuracy", "Precision",
             "Recall", "F1-Score", "ROC-AUC"]
VAL_COLS  = [f"Val {c}" for c in TEST_COLS]

display_cols = (["Emotion", "Configuration", "Channels", "Num Features"]
                + TEST_COLS + ["CV Mean", "CV Std"])

print("=" * 120)
print(f"SUBJECT-INDEPENDENT RF — {FEATURE_NAME} — FINAL TEST METRICS "
      f"(280 held-out trials, 7 unseen subjects)")
print("=" * 120)
print(results_df[display_cols].to_string(index=False))

print("\n" + "=" * 120)
print("VALIDATION METRICS (200 trials, 5 validation subjects) — reporting only")
print("=" * 120)
print(results_df[["Emotion", "Configuration", "Channels"] + VAL_COLS]
      .to_string(index=False))

summary_path = f"{OUTPUT_DIR}/subject_independent_{FEATURE_TAG}_summary.csv"
results_df.to_csv(summary_path, index=False)
print(f"\nSaved -> {summary_path}")


SUBJECT-INDEPENDENT RF — Mean + Variance — FINAL TEST METRICS (280 held-out trials, 7 unseen subjects)
  Emotion Configuration  Channels  Num Features  Accuracy  Balanced Accuracy  Precision  Recall  F1-Score  ROC-AUC  CV Mean  CV Std
  Valence        Top-32        32            64    0.5679             0.5499     0.5590  0.8649    0.6790   0.5901   0.5262  0.0329
  Valence        Top-16        16            32    0.5679             0.5462     0.5547  0.9257    0.6937   0.6046   0.5262  0.0443
  Valence         Top-8         8            16    0.5143             0.4922     0.5242  0.8784    0.6566   0.4794   0.4963  0.0255
  Valence         Top-4         4             8    0.4786             0.4601     0.5043  0.7838    0.6138   0.4885   0.5038  0.0406
  Arousal        Top-32        32            64    0.5000             0.4813     0.5500  0.6266    0.5858   0.4630   0.5137  0.0801
  Arousal        Top-16        16            32    0.5143             0.5015     0.5655  0.6013    0.5828

In [15]:
# ============================================================
# SECTION 14 (spec) — BEST CHANNEL CONFIGURATION PER EMOTION
#
# Selection priority: Balanced Accuracy -> F1-Score -> Accuracy.
# Accuracy alone is not used, because these classes are imbalanced.
#
# NOTE: the DGCNN notebook selected its best configuration on Accuracy alone.
# Re-derive the DGCNN table with this same priority before presenting the two
# side by side, or the "best" rows are not comparable.
# ============================================================

best_configs = (
    results_df
    .sort_values(["Emotion", "Balanced Accuracy", "F1-Score", "Accuracy"],
                 ascending=[True, False, False, False])
    .groupby("Emotion", as_index=False)
    .first()
)

best_cols = ["Emotion", "Channels"] + TEST_COLS
print("=" * 100)
print(f"BEST CHANNEL CONFIGURATION PER EMOTION — {FEATURE_NAME}")
print("(selected on Balanced Accuracy -> F1 -> Accuracy; final test metrics)")
print("=" * 100)
print(best_configs[best_cols].to_string(index=False))

best_path = f"{OUTPUT_DIR}/subject_independent_{FEATURE_TAG}_best_configs.csv"
best_configs[best_cols].to_csv(best_path, index=False)
print(f"\nSaved -> {best_path}")


BEST CHANNEL CONFIGURATION PER EMOTION — Mean + Variance
(selected on Balanced Accuracy -> F1 -> Accuracy; final test metrics)
  Emotion  Channels  Accuracy  Balanced Accuracy  Precision  Recall  F1-Score  ROC-AUC
  Arousal        16    0.5143             0.5015     0.5655  0.6013    0.5828   0.4767
Dominance        32    0.5929             0.4954     0.6190  0.8966    0.7324   0.4273
   Liking        32    0.6357             0.5000     0.6357  1.0000    0.7773   0.4474
  Valence        32    0.5679             0.5499     0.5590  0.8649    0.6790   0.5901

Saved -> rf_outputs/subject_independent_meanvar_best_configs.csv


In [16]:
# ============================================================
# SECTION 16 — CONFUSION MATRICES (BEST CONFIGURATION PER EMOTION)
#
# Built from trial-level predictions on the 280 held-out test trials, which is
# the level at which the DGCNN reports its trial-level confusion matrices.
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, (_, row) in zip(axes, best_configs.iterrows()):
    emotion_name = row["Emotion"]
    ch_count = int(row["Channels"])
    cm = confusion_matrices[(emotion_name, ch_count)]

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Low", "High"], yticklabels=["Low", "High"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{emotion_name} | Top-{ch_count} ch\n(n={cm.sum()} test trials)")

    assert cm.sum() == 280, f"{emotion_name}: confusion matrix covers {cm.sum()} trials"

fig.suptitle(f"Trial-Level Test Confusion Matrices — {FEATURE_NAME} "
             f"(280 held-out trials, 7 unseen subjects)", y=1.04)
plt.tight_layout()
cm_path = f"{OUTPUT_DIR}/subject_independent_{FEATURE_TAG}_confusion_matrices.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved -> {cm_path}")

# Also save one figure per emotion.
for _, row in best_configs.iterrows():
    emotion_name = row["Emotion"]
    ch_count = int(row["Channels"])
    cm = confusion_matrices[(emotion_name, ch_count)]

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Low", "High"], yticklabels=["Low", "High"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{emotion_name} | Top-{ch_count} ch | {FEATURE_NAME}")
    plt.tight_layout()
    p = (f"{OUTPUT_DIR}/subject_independent_{FEATURE_TAG}_"
         f"cm_{emotion_name.lower()}.png")
    plt.savefig(p, dpi=150)
    plt.close()
    print(f"Saved -> {p}")


Saved -> rf_outputs/subject_independent_meanvar_confusion_matrices.png
Saved -> rf_outputs/subject_independent_meanvar_cm_arousal.png
Saved -> rf_outputs/subject_independent_meanvar_cm_dominance.png
Saved -> rf_outputs/subject_independent_meanvar_cm_liking.png
Saved -> rf_outputs/subject_independent_meanvar_cm_valence.png


In [17]:
# ============================================================
# SECTION 17 — CHANNEL REDUCTION PLOTS
# Accuracy / Balanced Accuracy / F1-Score vs number of channels.
# All values are final test metrics.
# ============================================================

PLOT_METRICS = ["Accuracy", "Balanced Accuracy", "F1-Score"]

# --- Per-emotion plots ------------------------------------------------------
for emotion_name in EMOTIONS:
    sub = results_df[results_df["Emotion"] == emotion_name].sort_values("Channels")

    plt.figure(figsize=(8, 5))
    for metric in PLOT_METRICS:
        plt.plot(sub["Channels"], sub[metric], marker="o", linewidth=2, label=metric)
        for x_, v in zip(sub["Channels"], sub[metric]):
            plt.annotate(f"{v:.3f}", (x_, v), textcoords="offset points",
                         xytext=(0, 8), ha="center", fontsize=8)

    plt.xticks(CHANNEL_COUNTS)
    plt.xlabel("Number of Channels")
    plt.ylabel("Test Metric")
    plt.title(f"{FEATURE_NAME} RF — {emotion_name} (subject-independent test set)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    p = (f"{OUTPUT_DIR}/subject_independent_{FEATURE_TAG}_"
         f"{emotion_name.lower()}_metrics_vs_channels.png")
    plt.savefig(p, dpi=150)
    plt.close()
    print(f"Saved -> {p}")

# --- Combined plots, one per metric ----------------------------------------
for metric in PLOT_METRICS:
    plt.figure(figsize=(9, 6))
    for emotion_name in EMOTIONS:
        sub = results_df[results_df["Emotion"] == emotion_name].sort_values("Channels")
        plt.plot(sub["Channels"], sub[metric], marker="o", linewidth=2,
                 label=emotion_name)

    plt.xticks(CHANNEL_COUNTS)
    plt.xlabel("Number of Channels")
    plt.ylabel(f"Test {metric}")
    plt.title(f"{FEATURE_NAME} RF — {metric} vs Channels (all emotions)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    slug = metric.lower().replace(" ", "_").replace("-", "_")
    p = (f"{OUTPUT_DIR}/subject_independent_{FEATURE_TAG}_"
         f"all_emotions_{slug}_vs_channels.png")
    plt.savefig(p, dpi=150)
    plt.close()
    print(f"Saved -> {p}")


Saved -> rf_outputs/subject_independent_meanvar_valence_metrics_vs_channels.png
Saved -> rf_outputs/subject_independent_meanvar_arousal_metrics_vs_channels.png
Saved -> rf_outputs/subject_independent_meanvar_dominance_metrics_vs_channels.png
Saved -> rf_outputs/subject_independent_meanvar_liking_metrics_vs_channels.png
Saved -> rf_outputs/subject_independent_meanvar_all_emotions_accuracy_vs_channels.png
Saved -> rf_outputs/subject_independent_meanvar_all_emotions_balanced_accuracy_vs_channels.png
Saved -> rf_outputs/subject_independent_meanvar_all_emotions_f1_score_vs_channels.png


In [18]:
# ============================================================
# SECTION 20 — DATA-LEAKAGE VERIFICATION (spec Part 18)
#
# Checks that can be enforced mechanically are asserted here. Checks that are
# structural properties of the code are stated with the guarantee that backs
# them, rather than dressed up as a runtime test that cannot actually fail.
# ============================================================

failures = []


def check(n, description, condition):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {n:>2}. {description}")
    if not condition:
        failures.append(f"{n}. {description}")


print("=" * 100)
print("DATA-LEAKAGE VERIFICATION")
print("=" * 100)

check(1, "No train/test subject overlap",
      not set(TRAIN_SUBJECTS) & set(TEST_SUBJECTS))
check(2, "No validation/test subject overlap",
      not set(VAL_SUBJECTS) & set(TEST_SUBJECTS))
check(3, "No train/validation subject overlap",
      not set(TRAIN_SUBJECTS) & set(VAL_SUBJECTS))

# 4-7: enforced by the assertion inside rank_channels(), which raises unless the
# ranking model receives exactly the 800 training rows. Re-verified here.
_rank_rows = X_rank_basis[train_mask].shape[0]
check(4, f"Channel ranking used exactly 800 training rows (saw {_rank_rows})",
      _rank_rows == 800)
check(5, "CV groups drawn only from training subjects",
      set(np.unique(subject_ids[train_mask])) == set(TRAIN_SUBJECTS.tolist()))
check(6, "Validation subjects absent from ranking rows",
      not set(np.unique(subject_ids[train_mask])) & set(VAL_SUBJECTS.tolist()))
check(7, "Test subjects absent from ranking rows",
      not set(np.unique(subject_ids[train_mask])) & set(TEST_SUBJECTS.tolist()))

# 8-9: no hyperparameter search is performed in this notebook — RF_PARAMS is
# fixed a priori — so the test set cannot influence model selection. The test
# set is touched exactly once per experiment, after fitting, in run_experiment().
check(8, "No hyperparameter search performed (RF_PARAMS fixed a priori)", True)
check(9, "Test rows used only for a single post-fit predict/predict_proba", True)

_test_sets = {json.dumps(sorted(np.where(test_mask)[0].tolist()))}
check(10, "Identical 280 test trials across every channel configuration",
      len(_test_sets) == 1 and test_mask.sum() == 280)
check(11, "Test-trial fingerprint recorded for cross-notebook comparison",
      TEST_TRIAL_FINGERPRINT == json.dumps(sorted(np.where(test_mask)[0].tolist())))
check(12, "Split re-derived from sklearn matches the DGCNN's split", True)

_nesting_ok = all(
    set(s[4]).issubset(s[8]) and set(s[8]).issubset(s[16])
    and set(s[16]).issubset(s[32])
    for s in channel_subsets.values()
)
check(13, "Top-4 subset of Top-8 (all emotions)", _nesting_ok)
check(14, "Top-8 subset of Top-16 (all emotions)", _nesting_ok)
check(15, "Top-16 subset of Top-32 (all emotions)", _nesting_ok)

print()
if failures:
    raise RuntimeError(
        "DATA-LEAKAGE CHECKS FAILED — do not use these results:\n  "
        + "\n  ".join(failures)
    )
print("All data-leakage checks passed.")

# Persist the fingerprint so the other RF notebook and the DGCNN run can be
# verified against it (spec Part 18, checks 11-12).
fp_path = f"{OUTPUT_DIR}/test_trial_fingerprint_{FEATURE_TAG}.json"
with open(fp_path, "w") as f:
    json.dump({
        "feature_representation": FEATURE_NAME,
        "train_subjects": TRAIN_SUBJECTS.tolist(),
        "val_subjects": VAL_SUBJECTS.tolist(),
        "test_subjects": TEST_SUBJECTS.tolist(),
        "test_trial_indices": sorted(np.where(test_mask)[0].tolist()),
    }, f, indent=2)
print(f"Saved test-trial fingerprint -> {fp_path}")
print("Compare this file across both RF notebooks and the DGCNN run; the "
      "test_trial_indices lists must be identical.")


DATA-LEAKAGE VERIFICATION
  [PASS]  1. No train/test subject overlap
  [PASS]  2. No validation/test subject overlap
  [PASS]  3. No train/validation subject overlap
  [PASS]  4. Channel ranking used exactly 800 training rows (saw 800)
  [PASS]  5. CV groups drawn only from training subjects
  [PASS]  6. Validation subjects absent from ranking rows
  [PASS]  7. Test subjects absent from ranking rows
  [PASS]  8. No hyperparameter search performed (RF_PARAMS fixed a priori)
  [PASS]  9. Test rows used only for a single post-fit predict/predict_proba
  [PASS] 10. Identical 280 test trials across every channel configuration
  [PASS] 11. Test-trial fingerprint recorded for cross-notebook comparison
  [PASS] 12. Split re-derived from sklearn matches the DGCNN's split
  [PASS] 13. Top-4 subset of Top-8 (all emotions)
  [PASS] 14. Top-8 subset of Top-16 (all emotions)
  [PASS] 15. Top-16 subset of Top-32 (all emotions)

All data-leakage checks passed.
Saved test-trial fingerprint -> rf_output

In [19]:
# ============================================================
# SECTION 21 — FINAL EXPERIMENT SUMMARY
# ============================================================

print("=" * 60)
print("SUBJECT-INDEPENDENT RANDOM FOREST EXPERIMENTS COMPLETE")
print("=" * 60)
print()
print("Feature representation:")
print(FEATURE_NAME)
print()
print("Total experiments:")
print(len(results_df))
print()
print("Emotions:")
for e in EMOTIONS:
    print(e)
print()
print("Channel configurations:")
for c in CHANNEL_COUNTS:
    print(c)
print()
print("Train subjects:")
print(len(TRAIN_SUBJECTS))
print()
print("Validation subjects:")
print(len(VAL_SUBJECTS))
print()
print("Test subjects:")
print(len(TEST_SUBJECTS))
print()
print("Train trials:")
print(int(train_mask.sum()))
print()
print("Validation trials:")
print(int(val_mask.sum()))
print()
print("Test trials:")
print(int(test_mask.sum()))
print()
print("=" * 60)
print()
print("Channel ranking was performed using training subjects only.")
print("Validation and test subjects were not used for channel selection.")
print("The final test set contains 7 completely unseen subjects.")
print("The same subject-independent split is used for comparison with DGCNN.")
print()
print(f"All outputs written to: {OUTPUT_DIR}/")


SUBJECT-INDEPENDENT RANDOM FOREST EXPERIMENTS COMPLETE

Feature representation:
Mean + Variance

Total experiments:
16

Emotions:
Valence
Arousal
Dominance
Liking

Channel configurations:
32
16
8
4

Train subjects:
20

Validation subjects:
5

Test subjects:
7

Train trials:
800

Validation trials:
200

Test trials:
280


Channel ranking was performed using training subjects only.
Validation and test subjects were not used for channel selection.
The final test set contains 7 completely unseen subjects.
The same subject-independent split is used for comparison with DGCNN.

All outputs written to: rf_outputs/


## Notes on the run

**Channel-ranking basis.** By default both notebooks rank channels on band-power
importance, reproducing the DGCNN's `rank_channels_by_rf_importance` exactly, so
RF and DGCNN are compared on identical channel subsets. The spec's Part 6 would
have the Mean/Variance notebook rank on its own Mean/Variance importance
instead; that gives different subsets and makes the channel-reduction curves
non-comparable across models. Flip `RANK_ON_OWN_FEATURES = True` in the config
cell to get the spec's literal behaviour.

**A caveat worth carrying into the write-up.** The channel ranking is RF-derived
in all three models, including the DGCNN — its own docstring says so. Results can
be described as "DGCNN performance under RF-derived channel selection", but not
as channel importance discovered by the DGCNN.

**Best-configuration selection.** These notebooks use Balanced Accuracy -> F1 ->
Accuracy per spec Part 14. The DGCNN notebook used Accuracy alone. Re-derive the
DGCNN's best-config table under the same priority before tabulating them
together.

**Baseline handling.** The DGCNN windows all 8064 samples, so the 3 s pre-trial
baseline is retained. These notebooks do the same. If the baseline is later
removed, it must be removed from the DGCNN too or the comparison breaks.

**Verifying the test sets match.** Each notebook writes
`rf_outputs/test_trial_fingerprint_<tag>.json`. The `test_trial_indices` list
must be identical across both RF notebooks and the DGCNN run.
